# Submissão 3 — Modelo A (Stacking Ensemble: RoBERTa-base + DeBERTa-v3-base)

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import pickle
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import (
    RobertaTokenizer, RobertaForSequenceClassification,
    DebertaV2Tokenizer, DebertaV2ForSequenceClassification
)
from src.transformer_utils import InferenceDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

ID2LABEL_OUT = {0: 'Google', 1: 'Anthropic', 2: 'Meta', 3: 'OpenAI', 4: 'Human'}

device: cuda


## *carregar modelos e meta-classificador*

In [2]:
# RoBERTa-base
roberta_path  = '../models/model_roberta'
roberta_tok   = RobertaTokenizer.from_pretrained(roberta_path)
roberta_model = RobertaForSequenceClassification.from_pretrained(roberta_path).to(device)
roberta_model.eval()
print('RoBERTa-base carregado')

# DeBERTa-v3-base
deberta_path  = '../models/model_deberta'
deberta_tok   = DebertaV2Tokenizer.from_pretrained(deberta_path)
deberta_model = DebertaV2ForSequenceClassification.from_pretrained(deberta_path).float().to(device)
deberta_model.eval()
print('DeBERTa-v3-base carregado')

# Meta-classificador (SVM treinado no train_ensemble.ipynb)
with open('../models/ensemble_meta_clf.pkl', 'rb') as f:
    meta_clf = pickle.load(f)
print('meta-classificador carregado')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RoBERTa-base carregado


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DeBERTa-v3-base carregado
meta-classificador carregado


## *carregar dados*

In [3]:
df_subm = pd.read_csv('subm3.csv', sep=';')
texts   = df_subm['Text'].fillna('').tolist()
print(f'amostras a classificar: {len(texts)}')
print(df_subm.head(3))

amostras a classificar: 150
       ID                                               Text
0  D2-126  The reality about the places that diamonds are...
1  D2-127  Geothermobarometric calculations for a worldwi...
2  D2-128  Diamonds are formed deep within the Earth’s ma...


## *inferência*

In [4]:
MAX_LEN = 128

def get_probabilities(model, tokenizer, texts, batch_size=32):
    ds     = InferenceDataset(texts, tokenizer, MAX_LEN)
    loader = DataLoader(ds, batch_size=batch_size)
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            probs  = F.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)

print('a extrair probabilidades do RoBERTa...')
probs_r = get_probabilities(roberta_model, roberta_tok, texts)

print('a extrair probabilidades do DeBERTa...')
probs_d = get_probabilities(deberta_model, deberta_tok, texts)

X = np.hstack([probs_r, probs_d])
print(f'features: {X.shape}')

a extrair probabilidades do RoBERTa...
a extrair probabilidades do DeBERTa...
features: (150, 10)


In [5]:
preds = meta_clf.predict(X)
df_subm['Label'] = [ID2LABEL_OUT[p] for p in preds]
print(df_subm['Label'].value_counts())

Label
Human        50
Meta         50
Google       27
OpenAI       20
Anthropic     3
Name: count, dtype: int64


## *guardar submissão*

In [6]:
filename = 'subm3-g2-MEI-A.csv'
df_subm[['ID', 'Label']].to_csv(filename, index=False, sep=';')
print(f"Ficheiro '{filename}' guardado com sucesso!")
print(df_subm[['ID', 'Label']].head())

Ficheiro 'subm3-g2-MEI-A.csv' guardado com sucesso!
       ID  Label
0  D2-126  Human
1  D2-127  Human
2  D2-128  Human
3  D2-129  Human
4  D2-130   Meta
